# DSPy

In [1]:
import dspy
from typing import Literal
import litellm

In [2]:
model = "openai/gpt-5-mini"
lm = dspy.LM(model=model)
dspy.configure(lm=lm)

In [3]:
lm(messages=[
    {"role": "user", "content": "Hi"}
])

['Hi — how can I help you today? (I can answer questions, help with writing/code/math/planning, or anything else you need.)']

## What is Declarative?

### Case Study: Movie Review Sentiment

Requirements:
- `sentiment` (string, must be [`positive`, `negative`])
- `confidence` (float)

In [4]:
movie_review = """
The movie was boring and predictable. 
The plot was weak and the characters were one-dimensional. 
I would not recommend this movie to anyone.
"""

### Using API Call (LiteLLM)

In [5]:
def litellm_movie_review_sentiment(review: str):
    prompt = f"""
    This is a movie review: {review}
    Is the sentiment of this review positive or negative?
    Also mention the confidence score of your answer.
    """
    response = litellm.completion(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    response_content = response.choices[0].message.content.strip().lower()
    return response_content
    
litellm_movie_review_sentiment(movie_review)

'sentiment: negative.\n\nconfidence: 98% — the review uses clearly negative words ("boring," "predictable," "weak," "one-dimensional") and explicitly says "i would not recommend this movie."'

In [6]:
def litellm_movie_review_sentiment(review: str):
    system_prompt = """
You are a helpful assistant.
You will be given a movie review, and your task is to determine whether the sentiment of the review is positive or negative.
The review will be provided in the following format:
Review: <review text>
Your response should be either "positive", "negative", depending on the sentiment of the review.
You should also provide a confidence score for your answer, which should be a number between 0 and 1.
Response format:
Sentiment: <positive/negative>
Confidence: <confidence score>
"""

    user_prompt = f"""
Review: {review}
"""

    response = litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    response_content = response.choices[0].message.content.strip().lower()

    # extracting sentiment and confidence from the response
    sentiment_line = next(line for line in response_content.splitlines() if line.startswith("sentiment:"))
    sentiment = sentiment_line.split("sentiment:")[1].strip()
    confidence_line = next(line for line in response_content.splitlines() if line.startswith("confidence:"))
    confidence = float(confidence_line.split("confidence:")[1].strip())

    # check if sentiment is valid
    if sentiment not in ["positive", "negative"]:
        raise ValueError(f"Invalid sentiment: {sentiment}")

    return sentiment, confidence
    
litellm_movie_review_sentiment(movie_review)

('negative', 0.99)

### Using Declaritive (DSPy)

#### Signature -> Module -> Prediction

- Signature: how input and output should be formatted
- Module: how the pipeline works
- Prediction: output from the pipeline

In [7]:
class MovieSentimentSignature(dspy.Signature):
    """Extract the sentiment of a movie review"""
    review: str = dspy.InputField()
    sentiment: Literal["positive", "negative"] = dspy.OutputField()
    confidence: float = dspy.OutputField()

def dspy_movie_review_sentiment(movie_review: str):
    prediction_module = dspy.Predict(MovieSentimentSignature)
    return prediction_module(review=movie_review)

In [8]:
predication = dspy_movie_review_sentiment(movie_review)
predication

Prediction(
    sentiment='negative',
    confidence=0.98
)

In [9]:
predication.sentiment

'negative'

In [10]:
predication.confidence

0.98

#### Behind The Scene

In [11]:
dspy.inspect_history(n=1)





[2026-03-03T10:06:53.638601]

System message:

Your input fields are:
1. `review` (str):
Your output fields are:
1. `sentiment` (Literal['positive', 'negative']): 
2. `confidence` (float):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## review ## ]]
{review}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must exactly match (no extra characters) one of: positive; negative

[[ ## confidence ## ]]
{confidence}        # note: the value you produce must be a single float value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Extract the sentiment of a movie review


User message:

[[ ## review ## ]]

The movie was boring and predictable. 
The plot was weak and the characters were one-dimensional. 
I would not recommend this movie to anyone.


Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]` (must be formatted as a valid Python Lit

#### What if I want `neutral` class now?

- **LiteLLM**: Update prompt in several place + update output verification
- **DSPy**: Only update the Signature

In [12]:
class MovieSentimentSignature(dspy.Signature):
    """Extract the sentiment of a movie review"""
    review: str = dspy.InputField()
    sentiment: Literal["positive", "negative", "neutral"] = dspy.OutputField()    # <-- added "neutral" sentiment
    confidence: float = dspy.OutputField()

def dspy_movie_review_sentiment(movie_review: str):
    prediction_module = dspy.Predict(MovieSentimentSignature)
    return prediction_module(review=movie_review)

In [13]:
neutral_review = """The movie was okay."""
dspy_movie_review_sentiment(neutral_review)

Prediction(
    sentiment='neutral',
    confidence=0.82
)

#### What if I want chain-of-thought?

- **LiteLLM**: Update `litellm.completetion` to add `reasoning_effort`
- **DSPy**: Switch module from `dspy.Predict` to `dspy.ChainOfThought`

In [14]:
class MovieSentimentSignature(dspy.Signature):
    """Extract the sentiment of a movie review"""
    review: str = dspy.InputField()
    sentiment: Literal["positive", "negative", "neutral"] = dspy.OutputField()
    confidence: float = dspy.OutputField()

def dspy_movie_review_sentiment(movie_review: str):
    prediction_module = dspy.ChainOfThought(MovieSentimentSignature)  # <-- changed module to dspy.ChainOfThought
    return prediction_module(review=movie_review)

neutral_review = """The movie was okay."""
dspy_movie_review_sentiment(neutral_review)

Prediction(
    reasoning='The phrase "The movie was okay." expresses a lukewarm, middling opinion without clear praise or criticism. "Okay" conveys neutrality rather than distinctly positive or negative sentiment.',
    sentiment='neutral',
    confidence=0.95
)

In [15]:
dspy.inspect_history(n=1)





[2026-03-03T10:06:54.002422]

System message:

Your input fields are:
1. `review` (str):
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (Literal['positive', 'negative', 'neutral']): 
3. `confidence` (float):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## review ## ]]
{review}

[[ ## reasoning ## ]]
{reasoning}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must exactly match (no extra characters) one of: positive; negative; neutral

[[ ## confidence ## ]]
{confidence}        # note: the value you produce must be a single float value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Extract the sentiment of a movie review


User message:

[[ ## review ## ]]
The movie was okay.

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## sentiment ## ]]` (must be formatted as a valid Python Literal['positive', 

#### What if I want JSON (My model loves it)

- Just switch to JSON Adapter

In [16]:
dspy.configure(adapter=dspy.JSONAdapter())
dspy_movie_review_sentiment(neutral_review)

Prediction(
    reasoning='The phrase "The movie was okay." expresses a lukewarm, mildly positive but noncommittal appraisal without strong praise or criticism, corresponding to neutral sentiment.',
    sentiment='neutral',
    confidence=0.9
)

In [17]:
dspy.inspect_history(n=1)





[2026-03-03T10:06:54.128092]

System message:

Your input fields are:
1. `review` (str):
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (Literal['positive', 'negative', 'neutral']): 
3. `confidence` (float):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## review ## ]]
{review}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "sentiment": "{sentiment}        # note: the value you produce must exactly match (no extra characters) one of: positive; negative; neutral",
  "confidence": "{confidence}        # note: the value you produce must be a single float value"
}
In adhering to this structure, your objective is: 
        Extract the sentiment of a movie review


User message:

[[ ## review ## ]]
The movie was okay.

Respond with a JSON object in the following order of fields: `reasoning`, then `sentiment` (must be formatted as a

In [18]:
dspy.configure(adapter=dspy.ChatAdapter())

### Multi-stage Pipline

#### Sequential Pipeline: Email Processing

**Extract Metadata + Body $\rightarrow$ Analyze Sentiment**

In [19]:
email = """
From: "Peter Parker" <peter.parker@example.com>
To: "Stark Industries" <stark.industries@example.com>
Subject: Complaint about the new suit
Date: Wed, 1 Mar 2023 10:00:00 -0500

Dear Stark Industries,
I am writing to express my dissatisfaction with the new suit I purchased from your store. 
The suit does not fit well and the material feels cheap. I expected better quality from your brand.
Please let me know how you plan to address this issue.

- Peter Parker
"""

In [20]:
from datetime import datetime

In [21]:
class ExtractEmail(dspy.Signature):
    """Extract the sender, recipient, and body of an email from its raw text."""

    raw_email: str = dspy.InputField()
    sender_name: str = dspy.OutputField(desc="Last Name, First Name")           # <-- added description for names
    sender_email: str = dspy.OutputField()
    recipient_name: str = dspy.OutputField(desc="Last Name, First Name")
    recipient_email: str = dspy.OutputField()
    timestamp: datetime = dspy.OutputField()                                    # <-- added complex datatype
    body: str = dspy.OutputField()

class ClassifyEmailBody(dspy.Signature):
    """Classify sentiment of a given email body."""

    email_body: str = dspy.InputField()
    sentiment: Literal["positive", "negative", "neutral"] = dspy.OutputField()
    confidence: float = dspy.OutputField()

In [22]:
class EmailProcessor(dspy.Module):
    """A module to process emails."""
    def __init__(self):
        self.extract_email = dspy.Predict(ExtractEmail)
        self.classify_email_body = dspy.ChainOfThought(ClassifyEmailBody)

    def forward(self, raw_email: str):
        extraction_result = self.extract_email(raw_email=raw_email)
        classification_result = self.classify_email_body(email_body=extraction_result.body)
        
        return {
            "sender_name": extraction_result.sender_name,
            "sender_email": extraction_result.sender_email,
            "recipient_name": extraction_result.recipient_name,
            "recipient_email": extraction_result.recipient_email,
            "timestamp": extraction_result.timestamp,
            "body": extraction_result.body,
            "sentiment": classification_result.sentiment,
            "confidence": classification_result.confidence
        }

In [23]:
processor = EmailProcessor()
result = processor(raw_email=email)
result

{'sender_name': 'Parker, Peter',
 'sender_email': 'peter.parker@example.com',
 'recipient_name': 'Industries, Stark',
 'recipient_email': 'stark.industries@example.com',
 'timestamp': datetime.datetime(2023, 3, 1, 10, 0, tzinfo=TzInfo(-18000)),
 'body': 'Dear Stark Industries,\nI am writing to express my dissatisfaction with the new suit I purchased from your store. \nThe suit does not fit well and the material feels cheap. I expected better quality from your brand.\nPlease let me know how you plan to address this issue.\n\n- Peter Parker',
 'sentiment': 'negative',
 'confidence': 0.94}

In [24]:
# dspy.inspect_history(n=2)

#### Looped Pipeline: Article Writing

**Plan Outline + Sections + Subsections $\rightarrow$ Expand Sections**

In [27]:
class Outline(dspy.Signature):
    """Outline a thorough overview of a topic."""

    topic: str = dspy.InputField()
    title: str = dspy.OutputField()
    sections: list[str] = dspy.OutputField(desc="at most 5 sections")
    section_subheadings: dict[str, list[str]] = dspy.OutputField(desc="mapping from section headings to subheadings, at most 2 subheadings per section")

class DraftSection(dspy.Signature):
    """Draft a top-level section of an article."""

    topic: str = dspy.InputField()
    section_heading: str = dspy.InputField()
    section_subheadings: list[str] = dspy.InputField()
    content: str = dspy.OutputField(desc="markdown-formatted section, at most 50 words per sub-section")

class DraftArticle(dspy.Module):
    def __init__(self):
        self.build_outline = dspy.ChainOfThought(Outline)
        self.draft_section = dspy.ChainOfThought(DraftSection)


    def show_outline(self, outline):
        print('=== Outline ===')
        print(f'Title: {outline.title}')
        print('Sections:')
        for heading, subheadings in outline.section_subheadings.items():
            print(f'- {heading}')
            for subheading in subheadings:
                print(f'  - {subheading}')

    def show_drafting_progress(self, section_heading):
        print(f'=== Drafting section: {section_heading} ===')

    def forward(self, topic):
        outline = self.build_outline(topic=topic)
        self.show_outline(outline)

        sections = []
        for heading, subheadings in outline.section_subheadings.items():
            self.show_drafting_progress(heading)
            section, subheadings = f"## {heading}", [f"### {subheading}" for subheading in subheadings]
            section = self.draft_section(topic=outline.title, section_heading=section, section_subheadings=subheadings)
            sections.append(section.content)
        return dspy.Prediction(title=outline.title, sections=sections)

draft_article = DraftArticle()
article = draft_article(topic="History of UW-Madison")

=== Outline ===
Title: History of the University of Wisconsin–Madison: An Outline
Sections:
- Founding and early years (1848–1880)
  - State charter and establishment
  - Early campus, leadership, and curriculum
- Land-grant growth and the Wisconsin Idea (1880–1920)
  - Morrill Act influence and agricultural/technical programs
  - Origins and principles of the Wisconsin Idea (public service and outreach)
- Expansion, research, and academic leadership (1920–1960)
  - Growth of professional schools and research capacity
  - Federal research funding and wartime/postwar science
- Campus activism and social change (1960–1980)
  - Student movements, civil rights, and antiwar protests
  - Key incidents and institutional responses (campus governance, safety, and reform)
- Modernization, globalization, and contemporary challenges (1980–present)
  - Campus expansion, technology transfer, and global partnerships
  - Funding, governance, diversity initiatives, and free-speech debates
=== Drafting 

In [31]:
print(article.sections[0])

## Founding and early years (1848–1880)

### State charter and establishment
Chartered under Wisconsin’s 1848 state constitution and created by the legislature, the university was established as the state’s public institution. Governance by appointed regents and state support set its legal foundation; later Morrill Act land‑grant resources bolstered public funding and mission.

### Early campus, leadership, and curriculum
The early campus in Madison grew from modest buildings near the capitol area. Regents and early presidents prioritized a blend of classical liberal education, teacher training, and practical sciences (agriculture, engineering), shaping a broad public‑service academic mission.


## Tool Use

In [95]:
from search import wiki_search

In [96]:
class QASignature(dspy.Signature):
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

react = dspy.ReAct(QASignature, tools=[wiki_search])

In [104]:
# same as above but with string-based signature
react = dspy.ReAct("question -> answer", tools=[wiki_search])

In [99]:
react(question="What year was UW-Madison founded?")

Prediction(
    trajectory={'thought_0': "I'll search Wikipedia for the University of Wisconsin–Madison to confirm its founding year.", 'tool_name_0': 'wiki_search', 'tool_args_0': {'topic': 'University of Wisconsin–Madison'}, 'observation_0': 'The University of Wisconsin–Madison (University of Wisconsin, Wisconsin, UW, UW–Madison, or simply Madison) is a public land-grant research university in Madison, Wisconsin, United States. It was founded in 1848 when Wisconsin achieved statehood and is the flagship campus of the University of Wisconsin System. The 933-acre (378 ha) main campus is located on the shores of Lake Mendota; the university also owns and operates a 1,200-acre (486 ha) arboretum 4 miles (6.4 km) south of the main campus.\nUW–Madison is organized into 13 schools and colleges, which enrolled approximately 34,200 undergraduate and 14,300 graduate and professional students in 2024. Its academic programs include 136 undergraduate majors, 148 master\'s degree programs, and 120

## Pipeline Optimization

- Should I add some few-shot examples?
- Should I write detail description of tasks?
- Should I say "please" and "thank you"?
- Should I add "MAKE NO MISTAKE"?

In [100]:
from dspy.datasets import HotPotQA

In [101]:
trainset = [x.with_inputs('question') for x in HotPotQA(train_seed=2024, train_size=10).train]

In [ ]:
trainset[0]

Example({'question': 'Are Smyrnium and Nymania both types of plant?', 'answer': 'yes'}) (input_keys={'question'})

### Testing on Single Example

In [105]:
react(question=trainset[0]['question'])

Prediction(
    trajectory={'thought_0': 'I\'ll search Wikipedia for "Smyrnium" to confirm it\'s a plant genus, then search for "Nymania" to verify whether it\'s also a plant genus.', 'tool_name_0': 'wiki_search', 'tool_args_0': {'topic': 'Smyrnium'}, 'observation_0': 'Smyrnium is a genus of flowering plants in the family Apiaceae that naturally grows in a range stretching from Southern Europe to South Asia.\nSpecies include:\n\nSmyrnium cordifolium\nSmyrnium dodonaei Spreng.\nSmyrnium creticum\nSmyrnium olusatrum - alexanders, black-lovage, horse-parsley\nSmyrnium perfoliatum - perfoliate alexanders', 'thought_1': 'Search Wikipedia for "Nymania" to confirm whether it is a plant genus (and identify any species) so I can answer whether both are types of plant.', 'tool_name_1': 'wiki_search', 'tool_args_1': {'topic': 'Nymania'}, 'observation_1': 'Nymania capensis is a species of plant known in English as "Chinese lantern" because of the shape of its bright, colourful fruit, and in Afrika

### DSPy Optimizers
https://dspy.ai/learn/optimization/optimizers/

**Automatic Few-Shot Learning**
- LabeledFewShot
- BootstrapFewShot

**Automatic Instruction Optimization**
- COPRO
- MIPROv2
- GEPA

### MIPROv2

- Bootstrap Few-Shot Examples
- Propose Instruction Candidates
- Find an Optimized Combination of Few-Shot Examples & Instructions

In [ ]:
optimizer = dspy.MIPROv2(
    metric=dspy.evaluate.answer_exact_match,
    auto='light',
    num_threads=20,
)
optimized_react = optimizer.compile(react, trainset=trainset)

2026/03/03 12:28:50 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 20
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 8

2026/03/03 12:28:50 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/03/03 12:28:50 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/03/03 12:28:50 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 2/2 [00:01<00:00,  1.86it/s]


Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 4/6


100%|██████████| 2/2 [00:00<00:00,  2.00it/s]


...


Average Metric: 2.00 / 8 (25.0%): 100%|██████████| 8/8 [00:01<00:00,  4.03it/s]

2026/03/03 12:40:45 INFO dspy.evaluate.evaluate: Average Metric: 2 / 8 (25.0%)
2026/03/03 12:40:45 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 25.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0', 'Predictor 1: Instruction 0', 'Predictor 1: Few-Shot Set 2'].
2026/03/03 12:40:45 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [12.5, 0.0, 12.5, 0.0, 0.0, 0.0, 12.5, 12.5, 12.5, 0.0, 25.0, 25.0, 25.0, 0.0, 25.0]
2026/03/03 12:40:45 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 25.0
2026/03/03 12:40:45 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/03 12:40:45 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 16 / 20 =====



Average Metric: 2.00 / 8 (25.0%): 100%|██████████| 8/8 [00:14<00:00,  1.85s/it]

2026/03/03 12:41:00 INFO dspy.evaluate.evaluate: Average Metric: 2 / 8 (25.0%)
2026/03/03 12:41:00 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 25.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 4', 'Predictor 1: Instruction 0', 'Predictor 1: Few-Shot Set 3'].
2026/03/03 12:41:00 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [12.5, 0.0, 12.5, 0.0, 0.0, 0.0, 12.5, 12.5, 12.5, 0.0, 25.0, 25.0, 25.0, 0.0, 25.0, 25.0]
2026/03/03 12:41:00 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 25.0
2026/03/03 12:41:00 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/03 12:41:00 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 17 / 20 =====



Average Metric: 1.00 / 6 (16.7%):  75%|███████▌  | 6/8 [00:48<00:13,  6.64s/it]

In [ ]:
optimized_react